# Comparación: t-SNE, t-SNE cuántico, PCA clásico y qPCA

Este notebook compara técnicas clásicas y cuánticas simuladas para reducción de dimensionalidad usando el dataset Wine.

Configuraciones:
- `t-SNE clásico`: t-SNE sobre datos estandarizados.
- `t-SNE cuántico`: t-SNE usando distancias derivadas de fidelidades entre estados con amplitude encoding.
- `t-SNE + PCA clásico`: PCA clásico como reducción previa, luego t-SNE.
- `t-SNE + qPCA`: qPCA simulada vía matriz densidad/amplitude encoding, luego t-SNE.

> Nota metodológica: qPCA real requiere una forma eficiente de implementar $e^{i\rho t}$. Aquí se construye $\rho$ explícitamente para simular el flujo completo con Qiskit y mantener el experimento reproducible en máquina local.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display

# Evita errores de joblib/loky en Windows al detectar núcleos físicos.
# Debe ejecutarse antes de llamar algoritmos de sklearn que paralelizan internamente.
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    mean_squared_error,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)

from scipy.linalg import expm

from qiskit import QuantumCircuit
from qiskit.circuit.library import StatePreparation, UnitaryGate, phase_estimation
from qiskit.quantum_info import DensityMatrix, partial_trace, Pauli
from qiskit.visualization import plot_bloch_vector

RANDOM_STATE = 42
np.set_printoptions(precision=5, suppress=True)


## 1. Carga del dataset

La función `load_dataset()` deja aislada la carga de datos. Para cambiar Wine en el futuro, basta con modificar esta función y mantener la salida `(X, y, feature_names, target_name)`.

In [ ]:
def load_dataset():
    wine = fetch_ucirepo(id=109)
    X = wine.data.features.copy()
    y = wine.data.targets.iloc[:, 0].to_numpy()
    feature_names = list(X.columns)
    target_name = wine.data.targets.columns[0]
    return X, y, feature_names, target_name

X_df, y, feature_names, target_name = load_dataset()
X_raw = X_df.to_numpy(dtype=float)

print('Dataset:', X_raw.shape)
print('Target:', target_name)
print('Clases:', np.unique(y))
X_df.head()


## 2. Preprocesamiento consistente

Todos los métodos parten de los mismos datos estandarizados. Para amplitude encoding, además se aplica padding hasta la siguiente potencia de 2 y normalización L2 por muestra.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

def next_power_of_two(n):
    return 1 if n <= 1 else 2 ** math.ceil(math.log2(n))

def pad_to_dimension(X, target_dim):
    X_padded = np.zeros((X.shape[0], target_dim), dtype=float)
    X_padded[:, : X.shape[1]] = X
    return X_padded

def normalize_rows(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms

amplitude_dim = next_power_of_two(X_scaled.shape[1])
n_system_qubits = int(math.log2(amplitude_dim))

X_padded = pad_to_dimension(X_scaled, amplitude_dim)
X_amp = normalize_rows(X_padded)

print('Dimensión original:', X_scaled.shape[1])
print('Dimensión amplitude encoding:', amplitude_dim)
print('Qubits de sistema:', n_system_qubits)
print('Norma primera muestra:', np.linalg.norm(X_amp[0]))


## 4. PCA clásico para t-SNE

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

# Número de componentes PCA a usar para t-SNE
PCA_COMPONENTS_FOR_TSNE = min(8, X_scaled.shape[1])

# PCA clásico
pca = PCA(n_components=PCA_COMPONENTS_FOR_TSNE, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

# Para calcular el MSE de reconstrucción de PCA
X_pca_reconstructed = pca.inverse_transform(X_pca)
pca_variance = pca.explained_variance_ratio_.sum()
pca_mse = mean_squared_error(X_scaled, X_pca_reconstructed)

print(f'PCA clásico: Se usaron {PCA_COMPONENTS_FOR_TSNE} componentes.')
print(f'Varianza explicada por PCA clásico: {pca_variance:.4f}')
print(f'MSE de reconstrucción de PCA clásico: {pca_mse:.4f}')

In [ ]:
import matplotlib.pyplot as plt

# Calcular la varianza explicada acumulada
cum_variance_ratio = np.cumsum(pca.explained_variance_ratio_)

# Crear el gráfico
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cum_variance_ratio) + 1), cum_variance_ratio, marker='o', linestyle='--')
plt.xlabel('Número de Componentes Principales')
plt.ylabel('Varianza Explicada Acumulada')
plt.title('Varianza Explicada por el Número de Componentes Principales')
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(cum_variance_ratio) + 1))
plt.axhline(y=0.95, color='r', linestyle=':', label='95% de Varianza Explicada')
plt.legend()
plt.show()

Este gráfico muestra cómo la varianza explicada acumulada aumenta a medida que se añaden más componentes principales. Puedes observar dónde la curva empieza a aplanarse, lo que indica que añadir más componentes ya no aporta mucha más información. La línea roja punteada indica el 95% de la varianza explicada, un umbral común para decidir cuántos componentes conservar.

## 5. t-SNE sobre datos reducidos por PCA clásico

In [ ]:
from sklearn.manifold import TSNE

# t-SNE + PCA clásico
# La perplejidad se configura automáticamente en run_tsne
embedding_tsne_pca = run_tsne(X_pca, metric='euclidean', init='pca')

print('Embedding generado para t-SNE + PCA clásico:', embedding_tsne_pca.shape)

## 6. Tabla de métricas y evaluación

In [ ]:
def run_tsne(X, metric='euclidean', perplexity=30, init='random'):
    if perplexity is None:
        perplexity = min(30, max(5, (X.shape[0] - 1) // 3))
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        metric=metric,
        init=init,
        learning_rate='auto',
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    return tsne.fit_transform(X)

def embedding_metrics(embedding, labels):
    return {
        'Silhouette': silhouette_score(embedding, labels),
        'Harabach': calinski_harabasz_score(embedding, labels),
    }

def plot_embedding(ax, embedding, labels, title):
    scatter = ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=labels,
        cmap='viridis',
        s=38,
        alpha=0.85,
        edgecolor='k',
        linewidth=0.2,
    )
    ax.set_title(title)
    ax.set_xlabel('Dimensión 1')
    ax.set_ylabel('Dimensión 2')
    ax.grid(True, alpha=0.2)
    return scatter


In [ ]:
import pandas as pd
import numpy as np

results = []

def add_result(config, components, depth, width, variance, mse, embedding):
    metrics = embedding_metrics(embedding, y)
    results.append({
        'Configuración': config,
        'Componentes': components,
        'Profundidad': depth,
        'Ancho': width,
        'Varianza': variance,
        'MSE': mse,
        'Silhouette': metrics['Silhouette'],
        'Harabach': metrics['Harabach'],
    })

# Añadir resultado para t-SNE + PCA clásico
add_result('t-SNE + PCA clásico', PCA_COMPONENTS_FOR_TSNE, np.nan, np.nan, pca_variance, pca_mse, embedding_tsne_pca)

comparison_df = pd.DataFrame(results)
comparison_df_rounded = comparison_df.copy()
for col in ['Varianza', 'MSE', 'Silhouette', 'Harabach']:
    comparison_df_rounded[col] = comparison_df_rounded[col].astype(float).round(5)

display(comparison_df_rounded)

## 7. Visualización del Embedding `t-SNE + PCA clásico`

## 9. Experimentación con Parámetros de t-SNE

Vamos a ejecutar t-SNE nuevamente con un conjunto diferente de parámetros para ver cómo afecta la visualización y las métricas de clustering. En este caso, usaremos una perplejidad menor y una inicialización aleatoria para observar el impacto.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(8, 7), constrained_layout=True)

scatter = plot_embedding(ax, embedding_tsne_pca, y, 't-SNE + PCA clásico')

handles, _ = scatter.legend_elements()
fig.legend(handles, [str(c) for c in np.unique(y)], title=target_name, loc='center right', bbox_to_anchor=(1.08, 0.5))
plt.show()

In [ ]:
# Definir un rango de valores de perplejidad a probar
perplexity_values = [30, 40, 50, 60, 70]

# Asegurarse de que la lista de resultados esté limpia o inicializada correctamente
# Para este ejercicio, vamos a usar una nueva lista temporal para no interferir con los resultados anteriores directamente en 'results'
# O bien, si se desea continuar acumulando, asegurar que 'results' contenga los valores base.
# Para la comparación, lo más limpio es partir de los resultados ya calculados y añadir los nuevos.
# Asumiendo que 'results' ya contiene los resultados iniciales de t-SNE + PCA clásico

print("Ejecutando t-SNE con diferentes perplejidades...")

for p in perplexity_values:
    print(f"  - Perplejidad: {p}")
    embedding_tsne_pca_p = run_tsne(X_pca, metric='euclidean', perplexity=p, init='pca')
    add_result(
        f't-SNE + PCA (P={p}, Init=pca)',
        PCA_COMPONENTS_FOR_TSNE,
        np.nan,
        np.nan,
        pca_variance,
        pca_mse,
        embedding_tsne_pca_p
    )

# Mostrar la tabla de comparación actualizada
comparison_df_final = pd.DataFrame(results)
comparison_df_final_rounded = comparison_df_final.copy()
for col in ['Varianza', 'MSE', 'Silhouette', 'Harabach']:
    comparison_df_final_rounded[col] = comparison_df_final_rounded[col].astype(float).round(5)

display(comparison_df_final_rounded)

print("Comparación de t-SNE con diferentes perplejidades completada.")

Aquí tienes la tabla de métricas actualizada con los resultados de t-SNE para diferentes valores de perplejidad. Observa cómo cambian las métricas de `Silhouette` y `Harabach` a medida que modificamos este parámetro.

Generalmente, la perplejidad se puede interpretar como el número de vecinos cercanos que se consideran para cada punto. Un valor adecuado es crucial para un buen embedding, y este suele estar entre 5 y 50. Si es demasiado bajo, el embedding puede parecer muy fragmentado; si es demasiado alto, podría colapsar los clusters.

¿Puedes identificar alguna tendencia en las métricas con el cambio de perplejidad?

## 8. Análisis de Rendimiento: t-SNE + PCA clásico

Para evaluar la configuración `t-SNE + PCA clásico` y la idoneidad de sus parámetros, nos enfocaremos en las métricas disponibles en la tabla comparativa.

**Parámetros clave utilizados:**

*   **PCA Clásico:** Se eligieron **8 componentes** (`PCA_COMPONENTS_FOR_TSNE`).
*   **t-SNE:** La perplejidad (perplexity) se configuró automáticamente en **30** (calculado como `min(30, max(5, (X.shape[0] - 1) // 3))`), y la inicialización (`init`) fue `'pca'`.

**Interpretación de las métricas para `t-SNE + PCA clásico`:**

*   **`Componentes` (8):** Este valor indica que la PCA redujo los datos de 13 dimensiones originales a 8 antes de aplicar t-SNE. Es un hiperparámetro que equilibra la retención de información y la reducción de ruido. Un número adecuado de componentes es crucial para un buen rendimiento.

*   **`Varianza` (aproximadamente 0.92018):** Este valor (varianza explicada por PCA) indica que los 8 componentes principales retenidos por PCA capturan aproximadamente el **92.02% de la varianza total** de los datos originales. Un valor alto es deseable, ya que sugiere que la reducción de dimensionalidad conservó una gran parte de la información relevante, lo que es fundamental para que t-SNE pueda encontrar patrones significativos.

*   **`MSE` (aproximadamente 0.07982):** El Error Cuadrático Medio (MSE) mide la calidad de la reconstrucción de los datos originales a partir de los componentes PCA. Un MSE bajo (0.07982 en este caso) indica que los datos reconstruidos son muy similares a los datos originales. Esto refuerza la idea de que los 8 componentes son representativos y que la reducción de PCA es efectiva, minimizando la pérdida de información antes de t-SNE.

*   **`Silhouette` (aproximadamente 0.55631):** Este coeficiente evalúa la calidad de la agrupación de los clusters. Un valor más cercano a 1 indica que las muestras están bien agrupadas dentro de sus propios clusters y lejos de otros clusters. Un valor de 0.55631 es bastante bueno para el dataset Wine, sugiriendo una estructura de clusters razonablemente clara y bien definida en el embedding 2D generado por t-SNE. Esto implica que las clases originales se mantienen relativamente separadas y coherentes en la representación de menor dimensión.

*   **`Harabach` (aproximadamente 527.66632):** El índice Calinski-Harabasz es una métrica que busca clusters densos y bien separados. Valores más altos generalmente indican mejores clusters. Un valor de 527.66632, en conjunto con el coeficiente Silhouette, sugiere que el embedding de t-SNE + PCA clásico produce clusters que son razonablemente compactos y bien separados, lo que es un buen indicador de la calidad de la representación reducida.

*   **`Profundidad` (NaN) y `Ancho` (NaN):** Estas métricas son relevantes para los circuitos cuánticos y no aplican a los métodos clásicos como PCA y t-SNE, por lo que su valor es `NaN`.

**Conclusión sobre los parámetros:**

Los parámetros de **8 componentes para PCA** y una **perplejidad de 30 para t-SNE con inicialización 'pca'** parecen ser **muy adecuados** para el dataset Wine. La alta varianza explicada por PCA, el bajo MSE de reconstrucción, y las buenas métricas de clustering (Silhouette y Harabach) para el embedding final de t-SNE, sugieren que se ha logrado una reducción de dimensionalidad efectiva y una buena separación de las clases en el espacio 2D. La elección de iniciar t-SNE con 'pca' también es beneficiosa, ya que aprovecha la estructura lineal capturada por PCA, lo que a menudo lleva a mejores resultados y una convergencia más rápida para t-SNE.

## 3. Utilidades de evaluación y visualización